# UCSD EDA by Khwezi Kunene
Fantasy and Paranormal books. English reviews only


---

This dataset will explore the input files from https://cseweb.ucsd.edu/~jmcauley/datasets/goodreads.html

Fantasy & Paranormal (258,585 books, 55,397,550 interactions, 3,424,641 detailed reviews)
- goodreads_books_fantasy_paranormal.json.gz
- goodreads_interactions_fantasy_paranormal.json.gz
- goodreads_reviews_fantasy_paranormal.json.gz

## Imports and Data Loading

In [8]:
import json
import re
from collections import Counter
from tqdm import tqdm
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import nltk
nltk.download('vader_lexicon')
nltk.download('stopwords')
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.corpus import stopwords
import warnings
warnings.filterwarnings("ignore")

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /user/HS402/kk01697/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /user/HS402/kk01697/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [9]:
books_file = "/user/HS402/kk01697/Documents/dissertation/story-evaluation-dissertation/data/raw/goodreads/goodreads_books_fantasy_paranormal.json"
reviews_file = "/user/HS402/kk01697/Documents/dissertation/story-evaluation-dissertation/data/raw/goodreads/goodreads_reviews_fantasy_paranormal.json"

In [10]:
reviews = []

with open(reviews_file, "r") as f:
    for line in tqdm(f):
        reviews.append(json.loads(line))
review_df = pd.DataFrame(reviews)
print(review_df.shape)
review_df.head()

3424641it [00:09, 352235.46it/s]


(3424641, 11)


,user_id,book_id,review_id,rating,review_text,date_added,date_updated,read_at,started_at,n_votes,n_comments
0,8842281e1d1347389f2ab93d60773d4d,18245960,dfdbb7b0eb5a7e4c26d59a937e2e5feb,5,This is a special book. It started slow for ab...,Sun Jul 30 07:44:10 -0700 2017,Wed Aug 30 00:00:26 -0700 2017,Sat Aug 26 12:05:52 -0700 2017,Tue Aug 15 13:23:18 -0700 2017,28,1
1,8842281e1d1347389f2ab93d60773d4d,5577844,52c8ac49496c153e4a97161e36b2db55,5,A beautiful story. Neil Gaiman is truly a uniq...,Wed Sep 24 09:29:29 -0700 2014,Wed Oct 01 00:31:56 -0700 2014,Tue Sep 30 00:00:00 -0700 2014,Sun Sep 21 00:00:00 -0700 2014,5,1
2,8842281e1d1347389f2ab93d60773d4d,17315048,885c772fb033b041f42d57cef5be0a43,5,Mark Watney is a steely-eyed missile man. A ma...,Sat Apr 05 09:30:53 -0700 2014,Wed Mar 22 11:33:10 -0700 2017,Mon Aug 25 00:00:00 -0700 2014,Sat Aug 16 00:00:00 -0700 2014,25,5
3,8842281e1d1347389f2ab93d60773d4d,13453029,46a6e1a14e8afc82d221fec0a2bd3dd0,4,A fun fast paced book that sucks you in right ...,Tue Dec 04 11:12:22 -0800 2012,Sat Jul 26 11:43:28 -0700 2014,Tue Jul 08 00:00:00 -0700 2014,Wed Jul 02 00:00:00 -0700 2014,5,1
4,8842281e1d1347389f2ab93d60773d4d,13239822,a582bfa8efd69d453a5a21a678046b36,3,"This book has a great premise, and is full of ...",Mon Jul 02 16:04:16 -0700 2012,Wed Mar 22 11:32:20 -0700 2017,Wed Aug 15 00:00:00 -0700 2012,Sun Aug 12 00:00:00 -0700 2012,7,0


In [11]:
books = []

with open(books_file, "r") as f:
    for line in tqdm(f):
        books.append(json.loads(line))
book_df = pd.DataFrame(books)
print(book_df.shape)
book_df.head()

258585it [00:09, 26867.00it/s]


(258585, 29)


,isbn,text_reviews_count,series,country_code,language_code,popular_shelves,asin,is_ebook,average_rating,kindle_asin,...,publication_month,edition_information,publication_year,url,image_url,book_id,ratings_count,work_id,title,title_without_series
0,,7,[189911],US,eng,"[{'count': '58', 'name': 'to-read'}, {'count':...",B00071IKUY,false,4.03,,...,,Book Club Edition,1987,https://www.goodreads.com/book/show/7327624-th...,https://images.gr-assets.com/books/1304100136m...,7327624,140,8948723,"The Unschooled Wizard (Sun Wolf and Starhawk, ...","The Unschooled Wizard (Sun Wolf and Starhawk, ..."
1,1934876569,6,[151854],US,,"[{'count': '515', 'name': 'to-read'}, {'count'...",,false,4.22,,...,3,,2009,https://www.goodreads.com/book/show/6066812-al...,https://images.gr-assets.com/books/1316637798m...,6066812,98,701117,All's Fairy in Love and War (Avalon: Web of Ma...,All's Fairy in Love and War (Avalon: Web of Ma...
2,,60,[1052227],US,eng,"[{'count': '54', 'name': 'currently-reading'},...",B01NCIKAQX,true,4.33,B01NCIKAQX,...,,,,https://www.goodreads.com/book/show/33394837-t...,https://images.gr-assets.com/books/1493114742m...,33394837,269,54143148,The House of Memory (Pluto's Snitch #2),The House of Memory (Pluto's Snitch #2)
3,,1,[147734],US,,"[{'count': '1057', 'name': 'to-read'}, {'count...",B0056A00P4,true,4.04,B0056A00P4,...,,,,https://www.goodreads.com/book/show/12182387-t...,https://s.gr-assets.com/assets/nophoto/book/11...,12182387,4,285263,"The Passion (Dark Visions, #3)","The Passion (Dark Visions, #3)"
4,,21,[811663],US,en-US,"[{'count': '598', 'name': 'to-read'}, {'count'...",B01BLJGA9S,true,4.23,B01BLJGA9S,...,,,,https://www.goodreads.com/book/show/29074693-p...,https://s.gr-assets.com/assets/nophoto/book/11...,29074693,149,46079519,"Prowled Darkness (Dante's Circle, #7)","Prowled Darkness (Dante's Circle, #7)"


## Merge books and reviews

In [21]:
def merge(review_df, book_df):
    merged = review_df.merge(book_df, on='book_id', how='inner')
    print("Merged shape:", merged.shape)
    return merged

## Ratings and Reviews

In [ ]:
def rating_distribution(df):

    plt.figure(figsize=(7,5))
    sns.countplot(data=df, x='rating')

    plt.title("Review Ratings")
    plt.xlabel("Rating")
    plt.ylabel("Count")

    plt.show()

    print(df['rating'].describe())

In [ ]:
def review_length_analysis(df):
    df = df.copy()
    df['review_length'] = (df['review_text'].astype(str).str.split().str.len())
    print(df['review_length'].describe())
    
    plt.figure(figsize=(8,5))
    sns.histplot(df['review_length'],bins=50)

    plt.title("Review Length")
    plt.show()

    return df

In [ ]:
def review_length_vs_rating(df):
    plt.figure(figsize=(8,5))
    sns.boxplot(data=df, x='rating',y='review_length')
    plt.title("Review Length by Rating")
    plt.show()

In [ ]:
def popularity_analysis(df):
    plt.figure(figsize=(8,5))
    sns.histplot(np.log1p(df['ratings_count']),bins=40)
    plt.title("Book Popularity")
    plt.xlabel("Log Ratings Count")
    plt.show()
    print(df['ratings_count'].describe())

In [ ]:
def reviews_per_book(df):
    counts = (df.groupby('book_id').size())
    print(counts.describe())
    plt.figure(figsize=(8,5))
    sns.histplot(counts,bins=50)
    plt.xlim(0,100)
    plt.title("Reviews per Book")
    plt.show()

In [ ]:
def average_rating_analysis(df):
    plt.figure(figsize=(8,5))
    sns.histplot(df['average_rating'].astype(float), bins=30)

    plt.title("Book Average Rating")
    plt.show()

    print(df['average_rating'].describe())

In [ ]:
def correlation_analysis(df):
    cols = ['average_rating', 'ratings_count', 'text_reviews_count', 'n_votes', 'review_length']
    corr = (df[cols].corr())

    plt.figure(figsize=(8,6))

    sns.heatmap(corr,annot=True,cmap='Blues')

    plt.title("Feature Correlations")
    plt.show()

## Run EDA

In [ ]:
fan_df = merge(review_df, book_df)

In [ ]:
rating_distribution(fan_df)

In [ ]:
review_length_analysis(fan_df)

In [ ]:
reviews_per_book(fan_df)

In [ ]:
average_rating_analysis(fan_df)

In [ ]:
correlation_analysis(fan_df)